In [1]:
import pandas as pd

## Block 1: The mask (filtering rows)

- Filtering means keeping only the rows that pass a test.
- A comparison on a column (`students["marks"] > 80`) checks every row and gives a Series of True/False. This is called a mask.
- The mask has one True/False for every row, in the same order, with the same labels.
- `df[mask]` keeps only the rows where the mask is True.
- The mask is usually written inside the brackets: `df[df["marks"] > 80]`. The inner `df[...]` picks the column; the outer one filters.
- Comparison signs: `>`, `<`, `>=`, `<=`, `==` (equal), `!=` (not equal).
- Filtering gives a new table; the original `df` is unchanged.
- Labels are kept after filtering (Vivek stays row 1), so use `reset_index(drop=True)` if you want a fresh 0, 1, 2.
- A single `=` assigns; `==` compares. Writing `df["marks"] = 80` overwrites the column instead of testing it.

In [7]:

students = pd.read_csv("students.csv")     # name, city, marks (4 rows)
mask = students["marks"] > 80      # compare every mark with 80 -> True/False for each row
print(mask)                        # a Series of True/False, one per row
print(students[mask])              # keep only the rows where the mask is True
print(students[students["marks"] > 80])   # same thing, written in one line


0     True
1     True
2    False
3    False
Name: marks, dtype: bool
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91


## Block 2: Several conditions

- Combine masks with `&` (and), `|` (or) and `~` (not).
- `&`: a row stays only if both conditions are True. `|`: a row stays if at least one is True. `~` flips every True to False and every False to True.
- Write each condition in its own round brackets when they are inside one line: `df[(df["a"] > 1) & (df["b"] == "x")]`.
- Without brackets, `&` runs before the comparisons and the line fails.
- Python's words `and`, `or`, `not` do not work on masks; they give `ValueError: The truth value of a Series is ambiguous`. Use `&`, `|`, `~`.
- Saving each condition in its own named variable (`in_indore`, `high`) keeps long filters readable.
- A text comparison needs quotes and `==`: `df["city"] == "Indore"`. Capitals must match exactly.
- The result is a new table; labels are kept from the original.

In [11]:
indore = students["city"] == "Indore"
high = students["marks"] > 80
print(students[indore & high])
print(students[indore | high])
print(~indore)
print(students[(students["city"] == "Indore" )&(students["marks"]>80)])


   name    city  marks
0  Asha  Indore     82
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
3  Meena  Indore     76
0    False
1     True
2     True
3    False
Name: city, dtype: bool
   name    city  marks
0  Asha  Indore     82


## Block 3: isin and between

- `df["col"].isin([a, b, c])` gives True where the value is any item in the list. The list goes in square brackets.
- It replaces a long chain of `==` tests joined with `|`. Use `isin` as soon as you have more than two values.
- To get rows that are NOT in the list, put `~` in front: `~df["col"].isin([...])`.
- `df["col"].between(low, high)` gives True where `low <= value <= high`. Both end values are included.
- `between` is a shorter way to write `(df["col"] >= low) & (df["col"] <= high)`.
- Both give a mask, so they go inside `df[...]` like any other test, and can be combined with `&`, `|`, `~`.
- `isin("Pune")` with a plain string fails with TypeError; it needs a list: `isin(["Pune"])`.
- Text must match exactly, including capitals: `isin(["pune"])` finds nothing.

In [16]:
print(students["city"].isin(["Pune","Indore"]))
print(students[students["city"].isin(["Indore","Pune"])])
print(students[~students["city"].isin(["Pune", "Delhi"])]) 
print(students[(students["city"] == "Pune") | (students["city"] == "Delhi")])
print(students[students["marks"].between(75,85)])


0     True
1     True
2    False
3     True
Name: city, dtype: bool
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
3  Meena  Indore     76
    name    city  marks
0   Asha  Indore     82
3  Meena  Indore     76
    name   city  marks
1  Vivek   Pune     91
2   Ravi  Delhi     67
    name    city  marks
0   Asha  Indore     82
3  Meena  Indore     76


## Block 4: Missing values in filters

- `df["col"].isna()` gives True where the cell is empty (NaN); `df["col"].notna()` gives the opposite.
- Both give a mask, so they work inside `df[...]` and can be combined with `&`, `|`, `~`.
- `isna().sum()` counts the missing values (True counts as 1, False as 0). `df.isna().sum()` gives a count per column.
- A comparison with NaN is always False: `NaN > 1000`, `NaN == 5` and `NaN < 5` are all False.
- So `~(df["col"] > 1000)` also keeps the NaN rows, because NaN is not above 1000 and the NOT flips False to True.
- To avoid this, drop the missing rows first or test `notna()` in the same filter: `df[df["col"].notna() & (df["col"] > 1000)]`.
- `df["col"] == None` and `df["col"] == np.nan` do not find missing cells; use `isna()`.
- Filling or dropping missing values (`fillna`, `dropna`) comes in the cleaning topic.

In [22]:
orders = pd.read_csv("orders.csv") 
print(orders["amount"].isna())# empty cell true
print(orders["amount"].notna())
print(orders[orders["amount"].notna()].shape)
print(orders["amount"].isna().sum())
print(orders[orders["amount"].isna()].shape)

print(orders[orders["amount"] > 1000])                    # amount above 1000 -> 1001, 1004, 1007
print(orders[~(orders["amount"] > 1000)])                 # NOT above 1000 -> includes 1002 (NaN)! 5 rows
print(orders[orders["amount"].notna() & (orders["amount"] > 1000)]) # a clean filter, no NaN rows

0    False
1     True
2    False
3    False
4    False
5    False
6    False
7    False
Name: amount, dtype: bool
0     True
1    False
2     True
3     True
4     True
5     True
6     True
7     True
Name: amount, dtype: bool
(7, 5)
1
(1, 5)
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
3      1004    Meena  Indore  2300.0    NaN
6      1007    Arjun   Delhi  3100.0    5.0
   order_id customer    city  amount  items
1      1002    Vivek    Pune     NaN    1.0
2      1003     Ravi   Delhi   560.0    2.0
4      1005    Kabir  Mumbai   150.0    1.0
5      1006     Zoya    Pune   980.0    4.0
7      1008     Neha  Indore   450.0    2.0
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
3      1004    Meena  Indore  2300.0    NaN
6      1007    Arjun   Delhi  3100.0    5.0


In [31]:
names = orders["customer"]
print(names.str.startswith("A"))
print(orders[names.str.startswith("A")])
print(orders[names.str.endswith("a")])
print(orders[names.str.contains("a")])

print(orders[names.str.contains("a")])                    # lowercase a only: 6 rows, Arjun is NOT there
print(orders[names.str.contains("a", case=False)]) # case ignore

print(names.str.len())
print(orders[names.str.len()== 4])
print(orders[orders["city"].str.lower() == "pune"])


0     True
1    False
2    False
3    False
4    False
5    False
6     True
7    False
Name: customer, dtype: bool
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
6      1007    Arjun   Delhi  3100.0    5.0
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
3      1004    Meena  Indore  2300.0    NaN
5      1006     Zoya    Pune   980.0    4.0
7      1008     Neha  Indore   450.0    2.0
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
2      1003     Ravi   Delhi   560.0    2.0
3      1004    Meena  Indore  2300.0    NaN
4      1005    Kabir  Mumbai   150.0    1.0
5      1006     Zoya    Pune   980.0    4.0
7      1008     Neha  Indore   450.0    2.0
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
2      1003     Ravi   Delhi   560.0    2.0
3      1004    Meena  Indore  2300.0    NaN
4      1005    Kabir  Mumbai   150.0    1.0
5   

In [30]:
print(names.str.len())

0    4
1    5
2    4
3    5
4    5
5    4
6    5
7    4
Name: customer, dtype: int64


## Block 6: query()

- `df.query("condition")` filters rows by a condition written as one text string.
- Column names are written plainly, with no `df[...]` and no brackets: `df.query("amount > 1000")`.
- Use the words `and`, `or`, `not` inside the string, not `&`, `|`, `~`. No extra brackets are needed around each condition.
- A text value needs quotes inside the string. Use single quotes outside and double inside (or the reverse): `'city == "Indore"'`.
- `in [...]` works like `isin`: `'city in ["Pune", "Delhi"]'`.
- A normal Python variable is used with `@`: `df.query("amount > @limit")`.
- A column name with a space needs backticks: `` df.query("`Job Satisfaction` == 'High'") ``.
- A wrong or misspelled column name gives `UndefinedVariableError: name 'Amount' is not defined`; capitals must match.
- The result is the same as the mask way, with the same labels; it is a new table and the original is unchanged.
- Use query for short, readable filters; the mask way is more flexible (for example when the column name comes from a variable).

In [40]:
print(orders.query('city =="Indore" & amount > 1000'))
print(orders.query("amount > 1000")) 
print(orders.query('city == "Indore" or amount > 3000')) 
print(orders.query("city in ['Delhi','Pune']"))
print(orders.query('not (city == "Indore")'))



   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
3      1004    Meena  Indore  2300.0    NaN
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
3      1004    Meena  Indore  2300.0    NaN
6      1007    Arjun   Delhi  3100.0    5.0
   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
3      1004    Meena  Indore  2300.0    NaN
6      1007    Arjun   Delhi  3100.0    5.0
7      1008     Neha  Indore   450.0    2.0
   order_id customer   city  amount  items
1      1002    Vivek   Pune     NaN    1.0
2      1003     Ravi  Delhi   560.0    2.0
5      1006     Zoya   Pune   980.0    4.0
6      1007    Arjun  Delhi  3100.0    5.0
   order_id customer    city  amount  items
1      1002    Vivek    Pune     NaN    1.0
2      1003     Ravi   Delhi   560.0    2.0
4      1005    Kabir  Mumbai   150.0    1.0
5      1006     Zoya    Pune   980.0    4.0
6      1007    Arjun   Delhi  3100.0 

In [42]:
limit = 500
print(orders.query("amount > @limit"))
print(orders.query("amount.isna()"))

   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
2      1003     Ravi   Delhi   560.0    2.0
3      1004    Meena  Indore  2300.0    NaN
5      1006     Zoya    Pune   980.0    4.0
6      1007    Arjun   Delhi  3100.0    5.0
   order_id customer  city  amount  items
1      1002    Vivek  Pune     NaN    1.0


In [43]:
print(orders.query('customer.str.startswith("A")'))             # names starting with A -> 1001, 1007


   order_id customer    city  amount  items
0      1001     Asha  Indore  1200.0    3.0
6      1007    Arjun   Delhi  3100.0    5.0


In [44]:
a = orders.query("amount > 1000")
b = orders[orders["amount"] > 1000]
print(a.equals(b))   

True


## Block 7: loc with a mask

- `df.loc[rows, columns]` takes a row mask and the columns you want, in one step.
- Rows part: a mask (`df["amount"] > 1000`). Columns part: a name (gives a Series) or a list of names (gives a DataFrame).
- A mask alone, `df.loc[mask]`, keeps all columns, the same as `df[mask]`.
- A column range by name, `"customer":"amount"`, includes both ends and follows the table's column order.
- It is the fast way to answer questions like "total amount for Indore": `df.loc[df["city"] == "Indore", "amount"].sum()`.
- The row mask must be a mask of the same table. A mask built from another table gives an error or wrong rows.
- Labels are kept after filtering (0, 3, 6 stay 0, 3, 6). Use `reset_index(drop=True)` for a fresh 0, 1, 2.
- `df[mask]["col"]` and `df.loc[mask, "col"]` give the same values when you only read. For writing a value (next topic) you must use `loc`.
- A column that does not exist gives `KeyError`.
- Several conditions: wrap each in brackets and join with `&` or `|`, as in Block 2.

In [47]:
big = orders["amount"] > 100

print(orders.loc[big, "customer"])
print(orders.loc[big, ["customer", "amount"]]) 

0     Asha
2     Ravi
3    Meena
4    Kabir
5     Zoya
6    Arjun
7     Neha
Name: customer, dtype: object
  customer  amount
0     Asha  1200.0
2     Ravi   560.0
3    Meena  2300.0
4    Kabir   150.0
5     Zoya   980.0
6    Arjun  3100.0
7     Neha   450.0


In [48]:
print(orders.loc[big , "customer": "amount"])

  customer    city  amount
0     Asha  Indore  1200.0
2     Ravi   Delhi   560.0
3    Meena  Indore  2300.0
4    Kabir  Mumbai   150.0
5     Zoya    Pune   980.0
6    Arjun   Delhi  3100.0
7     Neha  Indore   450.0


In [52]:
print(orders.loc[orders["city"]=="Indore", "amount"].sum())
orders.loc[orders["city"]=="Indore", "amount"].mean()

3950.0


1316.6666666666667

In [62]:
print(orders.loc[(orders.city == "Indore") & (orders.amount > 1000) , ["order_id", "amount"]])

   order_id  amount
0      1001  1200.0
3      1004  2300.0


In [63]:
print(orders.loc[big].shape)
print(orders.loc[big, "amount"].isna().shape)

print(orders.loc[big, "amount"].shape)     # (3,)    one name -> Series
print(orders.loc[big, ["amount"]].shape)   # (3, 1)  list of one name -> DataFrame


(7, 5)
(7,)
(7,)
(7, 1)
